# 📖 Notebook 4: Security Gates in CI/CD

You've learned to find vulnerabilities manually. But what about the hundreds of developers on your team? You can't review every line of code by hand.

The answer is **automated security gates** — tools that run on every pull request and block insecure code from being merged.

At Microsoft, the SDL requires that **no code ships without passing automated security checks**. This notebook shows you what those checks look like.

## Learning Objectives

By the end of this notebook, you'll understand:
- What SAST (Static Analysis) is and how to run it
- What DAST (Dynamic Analysis) is and how it differs from SAST
- How to check dependencies for known vulnerabilities
- How to set up a security gate pipeline
- The security sign-off process at large enterprises

## 🛠️ Setup

```bash
cd 08-enterprise/security-review
docker compose up -d
source .venv/bin/activate
uv sync
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

## The Security Gate Pipeline

In a modern CI/CD pipeline, security checks run automatically:

```
Developer pushes code
        │
        ▼
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│  Gate 1: SAST   │────▶│  Gate 2: Deps   │────▶│  Gate 3: DAST   │
│                 │     │                 │     │                 │
│ Scan source     │     │ Check libraries │     │ Scan the running│
│ code for bugs   │     │ for known CVEs  │     │ app for vulns   │
│                 │     │                 │     │                 │
│ Tool: Bandit    │     │ Tool: pip-audit │     │ Tool: OWASP ZAP │
│       Semgrep   │     │       Dependabot│     │       Burp Suite│
└────────┬────────┘     └────────┬────────┘     └────────┬────────┘
         │                       │                       │
    Pass/Fail               Pass/Fail               Pass/Fail
         │                       │                       │
         └───────────────────────┼───────────────────────┘
                                 ▼
                        ┌────────────────┐
                        │ All gates pass │──▶ ✅ Merge allowed
                        │ Any gate fails │──▶ ❌ Merge blocked
                        └────────────────┘
                                 │
                                 ▼
                        ┌────────────────┐
                        │ Gate 4: Human  │
                        │ Security       │
                        │ Review         │
                        │ (for critical  │
                        │  changes)      │
                        └────────────────┘
```

## Gate 1: SAST — Static Application Security Testing

**SAST** scans your **source code** without running it. It looks for patterns that are known to be dangerous.

| Feature | Details |
|---------|--------|
| **When it runs** | On every commit / pull request |
| **What it scans** | Source code, configuration files |
| **What it finds** | SQL injection, hardcoded secrets, insecure functions |
| **Speed** | Fast (seconds to minutes) |
| **False positives** | Moderate — some findings may not be exploitable |

### Popular SAST Tools

| Tool | Language | Free? |
|------|----------|-------|
| **Bandit** | Python | ✅ Open source |
| **Semgrep** | Many languages | ✅ Community rules |
| **CodeQL** | Many languages | ✅ Free for open source (GitHub) |
| **SonarQube** | Many languages | ✅ Community edition |
| **Checkmarx** | Many languages | ❌ Commercial |

In [ ]:
# === Gate 1: Run Bandit (Python SAST tool) on our vulnerable app ===
# Bandit is the most popular open-source SAST tool for Python.
# It checks for common security issues like:
# - SQL injection (B608)
# - Hardcoded passwords (B105, B106)
# - Use of exec/eval (B102)
# - Weak cryptography (B324)
# - Flask debug mode (B201)

import json
import subprocess
import sys

print("🔍 Gate 1: SAST — Running Bandit on vulnerable_app.py")
print("=" * 70)

result = subprocess.run(
    [sys.executable, "-m", "bandit", "-r", "../app/vulnerable_app.py", "-f", "json"],
    capture_output=True,
    text=True,
)

bandit_high = bandit_medium = bandit_low = 0
bandit_results = []
try:
    report = json.loads(result.stdout)
    bandit_results = report.get("results", [])

    print(f"\nFound {len(bandit_results)} security issues:\n")

    for issue in bandit_results:
        severity = issue["issue_severity"]
        confidence = issue["issue_confidence"]
        icon = {"HIGH": "🔴", "MEDIUM": "🟡", "LOW": "🟢"}.get(severity, "⚪")

        print(f"  {icon} [{severity}/{confidence}] {issue['test_id']}: {issue['issue_text']}")
        print(f"     Line {issue['line_number']}: {issue['code'].strip().split(chr(10))[0]}")
        print()

    # Summary — keep these around, the pipeline report at the end reuses them.
    metrics = report.get("metrics", {}).get("_totals", {})
    bandit_high = metrics.get("SEVERITY.HIGH", 0)
    bandit_medium = metrics.get("SEVERITY.MEDIUM", 0)
    bandit_low = metrics.get("SEVERITY.LOW", 0)

    print(f"Summary: {bandit_high} High, {bandit_medium} Medium, {bandit_low} Low")

    if bandit_high > 0:
        print("\n❌ GATE FAILED: High severity issues found. Merge blocked.")
    else:
        print("\n✅ GATE PASSED: No high severity issues.")

except json.JSONDecodeError:
    print("Bandit output:")
    print(result.stdout or result.stderr)

assert bandit_results, (
    "Bandit reported nothing against a deliberately vulnerable file — it did not "
    "actually run (check `uv sync` and that you are in notebooks/)"
)
assert bandit_high > 0, (
    f"expected at least one HIGH finding so this gate demonstrates a FAIL, got "
    f"{bandit_high}"
)

# --- Now the part a SAST demo usually skips: what did it MISS? --------------
found_tests = {i["test_id"] for i in bandit_results}
print("\n" + "=" * 70)
print("🕳️  What Bandit did NOT find in this file")
print("=" * 70)
assert "B608" in found_tests, "Bandit stopped catching the SQL injection (B608)"
print("  ✅ caught: the SQL injection (B608) and the MD5 hash (B324)")
print("  ❌ missed: the stored XSS in /comments/<id>")
print("  ❌ missed: the SSTI — render_template_string() on a string built from")
print("             user input. There is no Bandit check for it.")
print("  ❌ missed: the CSRF hole in /api/transfer (no token check)")
print("  ❌ missed: the SSRF in /api/fetch-url (fetching a user-supplied URL)")
print("  ❌ missed: API_KEY and DATABASE_URL — B105 matched SECRET_KEY only,")
print("             because its rule looks for the word 'secret', not 'key'.")
print("\nSAST matches patterns in code it can see. Every one of those misses is a")
print("bug about how data FLOWS at runtime, or a rule nobody wrote. That is not a")
print("knock on Bandit — it is the entire reason Gate 3 (DAST) exists, and why")
print("'the scan is green' is a starting point for a review, never its conclusion.")

### Understanding Bandit Results

Each Bandit finding has:
- **Test ID** (e.g., B608): Identifies the specific check
- **Severity**: How dangerous the issue is (HIGH/MEDIUM/LOW)
- **Confidence**: How sure Bandit is that this is a real issue (HIGH/MEDIUM/LOW)

Common Bandit checks relevant to our app:

| ID | Check | What It Finds |
|----|-------|---------------|
| B105 | hardcoded_password_string | Passwords assigned to variables |
| B106 | hardcoded_password_funcarg | Passwords passed to function arguments |
| B324 | hashlib | Use of a weak hash (MD5/SHA1) for security |
| B201 | flask_debug_true | `app.run(debug=True)` — exposes the Werkzeug console |
| B104 | hardcoded_bind_all_interfaces | Binding to `0.0.0.0` |
| B608 | hardcoded_sql_expressions | SQL built with string formatting |

> Bandit's older docs call the MD5 check **B303**; current versions report it as
> **B324** (`hashlib`). If you pin a rule ID in a CI config, pin the one your
> installed version actually emits — a gate that references a retired ID silently
> checks nothing.

## Gate 2: Dependency Vulnerability Scanning

Your code might be secure, but **what about your dependencies?** If you're using a library with a known vulnerability, attackers can exploit it.

The famous **Log4Shell** vulnerability (CVE-2021-44228) affected millions of applications through a single Java logging library.

### How It Works

1. Tools read your dependency file (`pyproject.toml`, `package.json`, etc.)
2. They check each package version against databases of known vulnerabilities (CVEs)
3. If a vulnerable version is found, the build fails

### Popular Dependency Scanners

| Tool | Language | Free? |
|------|----------|-------|
| **pip-audit** | Python | ✅ PyPA-maintained, no account needed (recommended) |
| **safety** | Python | ⚠️ Free tier deprecated; needs an account |
| **npm audit** | JavaScript | ✅ Built-in |
| **Dependabot** | Many | ✅ GitHub built-in |
| **Snyk** | Many | ✅ Free tier |

In [ ]:
# === Gate 2: Dependency vulnerability scanning ===
# Let's scan our pyproject.toml for known vulnerabilities

# First, let's look at what we're scanning
print("📦 Our dependencies (pyproject.toml):")
print("=" * 50)
with open("../pyproject.toml") as f:
    deps = f.read()
    print(deps)

print("\n🔍 Gate 2: Scanning dependencies for known vulnerabilities...")
print("=" * 70)

In [ ]:
# Run pip-audit (modern successor to safety) on installed packages.
# pip-audit uses the OSV and PyPI advisory databases and is maintained by PyPA.
import subprocess
import sys

print("Running 'pip-audit' on the current environment...\n")

result = subprocess.run(
    [sys.executable, "-m", "pip_audit", "--progress-spinner", "off"],
    capture_output=True,
    text=True,
)

output = result.stdout + result.stderr
print(output[:2000])  # show first 2000 chars

deps_clean = result.returncode == 0
if deps_clean:
    print("\n✅ GATE PASSED: No known vulnerabilities in dependencies.")
else:
    print("\n⚠️  Vulnerabilities found! In a real pipeline, this would block the merge.")
    print("   Fix: Update affected packages to patched versions.")

# This gate's result is a moving target: it depends on what CVEs were published
# this week, not on anything in this repo. So do NOT assert it passes -- assert
# only that the tool actually ran and produced a verdict we can act on.
assert result.returncode is not None and (result.stdout or result.stderr), (
    "pip-audit produced no output at all -- it did not run (try `uv sync`)"
)
print(f"\n(pip-audit exit code {result.returncode}; a dependency gate is the one "
      f"gate whose\n result can change overnight without anyone touching the code.)")

# Note: the older `safety check` command is deprecated; pip-audit is the
# recommended replacement for beginners because it needs no account or API key.


In [ ]:
# === DEMO: What a vulnerable dependency looks like ===
# Let's simulate finding a vulnerability in a dependency

simulated_vulns = [
    {
        "package": "flask",
        "installed": "2.3.0",
        "affected": "<2.3.2",
        "fixed_in": "2.3.2",
        "cve": "CVE-2023-30861",
        "severity": "HIGH",
        "description": "Cookie handling could allow session data leakage",
    },
    {
        "package": "requests",
        "installed": "2.28.0",
        "affected": "<2.31.0",
        "fixed_in": "2.31.0",
        "cve": "CVE-2023-32681",
        "severity": "MEDIUM",
        "description": "Proxy-Authorization header leaked to third-party hosts",
    },
]

print("📊 Example: Dependency Vulnerability Report")
print("=" * 70)
print("(Simulated — real results depend on your installed versions)\n")

for vuln in simulated_vulns:
    icon = "🔴" if vuln["severity"] == "HIGH" else "🟡"
    print(f"  {icon} {vuln['package']} {vuln['installed']}")
    print(f"     CVE: {vuln['cve']} ({vuln['severity']})")
    print(f"     Issue: {vuln['description']}")
    print(f"     Affected: {vuln['affected']}")
    print(f"     Fix: pip install {vuln['package']}>={vuln['fixed_in']}")
    print()

print("💡 Dependabot (built into GitHub) can automatically create PRs")
print("   to update vulnerable packages.")

## Gate 3: DAST — Dynamic Application Security Testing

**DAST** tests your **running application** by sending malicious requests and checking the responses.

| Feature | SAST | DAST |
|---------|------|------|
| **Scans** | Source code | Running application |
| **Requires** | Code access | Network access |
| **Finds** | Code-level bugs | Runtime vulnerabilities |
| **Speed** | Fast | Slower |
| **False positives** | Higher | Lower |
| **Example tools** | Bandit, Semgrep | OWASP ZAP, Burp Suite |

DAST is like hiring a robot pentester that automatically tries common attacks against your app.

In [ ]:
# === Gate 3: Simple DAST — automated vulnerability testing ===
# In production you'd use OWASP ZAP or Burp Suite.
# Here we build a simple scanner to demonstrate the concept.

import requests

BASE_URL = "http://localhost:5001"


def _json(resp):
    """Response bodies from a crashing endpoint are HTML, not JSON."""
    try:
        return resp.json()
    except ValueError:
        return None


def _reset_login_limits():
    """Clear /api/login/safe's Redis rate-limit counters so repeated runs of this
    scanner see 401s rather than 429s."""
    try:
        import redis
        r = redis.Redis(host="localhost", port=6379, decode_responses=True)
        stale = r.keys("login_attempts:*")
        if stale:
            r.delete(*stale)
    except Exception:
        pass


class SimpleDAST:
    """A minimal DAST scanner for educational purposes.
    Real tools like OWASP ZAP have thousands of checks.

    Every check probes BOTH the vulnerable endpoint and its /safe twin. The
    /safe probe is the control: if it ever produces a finding, a fix regressed.
    A control you skip -- for example by `break`ing out of the loop as soon as
    the vulnerable endpoint reports a hit -- is not a control at all, because it
    can never fail. That is worth more attention than it usually gets: a check
    that cannot fail looks exactly like a check that passes.
    """

    # Non-destructive payloads only. `'; DROP TABLE products; --` would be a
    # realistic payload and psycopg2 really does execute multiple statements --
    # which is precisely why a scanner should never fire it anywhere it is not
    # certain the data is disposable.
    SQLI_PAYLOADS = [
        "' OR '1'='1' --",
        "' OR 1=1 --",
        "' UNION SELECT id, name, price FROM products --",
    ]

    def __init__(self, base_url):
        self.base_url = base_url
        self.findings = []

    def _add(self, type_, severity, endpoint, payload, evidence):
        self.findings.append({
            "type": type_, "severity": severity, "endpoint": endpoint,
            "payload": payload, "evidence": evidence,
        })

    # ------------------------------------------------------------------ SQLi
    def test_sql_injection(self):
        """Test for SQL injection vulnerabilities."""
        baseline = len(_json(requests.get(
            f"{self.base_url}/api/products/search", params={"q": "Laptop"})) or [])

        for payload in self.SQLI_PAYLOADS:
            rows = _json(requests.get(
                f"{self.base_url}/api/products/search", params={"q": payload}))
            if rows is not None and len(rows) > baseline:
                self._add("SQL Injection", "CRITICAL", "/api/products/search", payload,
                          f"Returned {len(rows)} rows vs {baseline} for an honest search")
                break  # one finding on the vulnerable endpoint is enough

        # CONTROL: every payload, every time, against the parameterized endpoint.
        for payload in self.SQLI_PAYLOADS:
            rows = _json(requests.get(
                f"{self.base_url}/api/products/search/safe", params={"q": payload}))
            if rows is not None and len(rows) > baseline:
                self._add("SQL Injection", "CRITICAL", "/api/products/search/safe",
                          payload, f"Safe endpoint returned {len(rows)} rows — REGRESSION")

    # ------------------------------------------------------------- XSS / SSTI
    def test_xss_and_ssti(self):
        """Test for stored XSS and for server-side template injection."""
        xss_payload = "<script>alert(1)</script>"
        # 7 digits on purpose: the page also renders each comment's timestamp,
        # whose longest digit run is the 6-digit microseconds field, so this
        # marker cannot appear by coincidence.
        ssti_payload = "{{7*191*1000}}"
        ssti_marker = "1337000"

        for payload in (xss_payload, ssti_payload):
            requests.post(f"{self.base_url}/api/comments", json={
                "user_id": 2, "product_id": 1, "content": payload,
            })

        vulnerable = requests.get(f"{self.base_url}/comments/1").text
        if "<script>" in vulnerable:
            self._add("Stored XSS", "HIGH", "/comments/1", xss_payload,
                      "Unescaped <script> tag found in response")
        if ssti_marker in vulnerable:
            self._add("Server-Side Template Injection", "CRITICAL", "/comments/1",
                      ssti_payload,
                      f"{ssti_payload} came back as {ssti_marker} — evaluated on the server")

        # CONTROL: the escaped endpoint must neutralise BOTH.
        safe = requests.get(f"{self.base_url}/comments/1/safe").text
        if "<script>" in safe:
            self._add("Stored XSS", "HIGH", "/comments/1/safe", xss_payload,
                      "Safe endpoint emitted an unescaped script tag — REGRESSION")
        if ssti_marker in safe:
            self._add("Server-Side Template Injection", "CRITICAL", "/comments/1/safe",
                      ssti_payload,
                      "Safe endpoint EVALUATED the template — escaping alone never "
                      "fixed SSTI — REGRESSION")

    # ------------------------------------------------------------------ SSRF
    def test_ssrf(self):
        """Test for Server-Side Request Forgery."""
        # docker-compose service names: reachable from the app container, not
        # from this notebook. That asymmetry is what makes SSRF worth something.
        internal_urls = [
            "http://localhost:5001/health",
            "http://adminer:8080/",
        ]

        for url in internal_urls:
            data = _json(requests.get(f"{self.base_url}/api/fetch-url",
                                      params={"url": url}, timeout=20))
            if data and "status_code" in data:
                self._add("SSRF", "CRITICAL", "/api/fetch-url", url,
                          f"Server fetched an internal URL (status {data['status_code']})")
                break

        # CONTROL: the allowlisted endpoint must refuse internal targets. Probe
        # an https:// one too, so we exercise the private-IP check and not just
        # the cheaper "HTTPS only" rejection.
        for url in internal_urls + ["https://localhost:5001/health"]:
            resp = requests.get(f"{self.base_url}/api/fetch-url/safe",
                                params={"url": url}, timeout=20)
            if resp.status_code not in (400, 403):
                self._add("SSRF", "CRITICAL", "/api/fetch-url/safe", url,
                          f"Safe endpoint answered {resp.status_code} for an internal "
                          f"URL — REGRESSION")

    # ------------------------------------------------------------------ CSRF
    def test_csrf(self):
        """Test for missing CSRF protection."""
        # form-encoded, because that is the only body a cross-site <form> can send
        body = {"from_user": "alice", "to_user": "attacker", "amount": "100"}
        headers = {"Origin": "http://evil.com"}

        resp = requests.post(f"{self.base_url}/api/transfer", data=body, headers=headers)
        if resp.status_code == 200:
            self._add("Missing CSRF Protection", "HIGH", "/api/transfer",
                      "Cross-origin form POST with no CSRF token",
                      "Transfer succeeded from an evil.com Origin")

        # CONTROL
        resp = requests.post(f"{self.base_url}/api/transfer/safe", data=body,
                             headers=headers)
        if resp.status_code != 403:
            self._add("Missing CSRF Protection", "HIGH", "/api/transfer/safe",
                      "Cross-origin form POST with no CSRF token",
                      f"Safe endpoint answered {resp.status_code} — REGRESSION")

    # -------------------------------------------------------- info disclosure
    def test_info_disclosure(self):
        """Test for user enumeration on login."""
        _reset_login_limits()
        real = requests.post(f"{self.base_url}/api/login",
                             json={"username": "alice", "password": "wrong"})
        fake = requests.post(f"{self.base_url}/api/login",
                             json={"username": "nonexistent_xyz", "password": "wrong"})
        if real.status_code != fake.status_code:
            self._add("Information Disclosure", "MEDIUM", "/api/login",
                      "Different error for existing vs non-existing user",
                      f"Existing user: {real.status_code}, Non-existing: {fake.status_code}")

        # CONTROL
        _reset_login_limits()
        real = requests.post(f"{self.base_url}/api/login/safe",
                             json={"username": "alice", "password": "wrong"})
        fake = requests.post(f"{self.base_url}/api/login/safe",
                             json={"username": "nonexistent_xyz", "password": "wrong"})
        if (real.status_code, _json(real)) != (fake.status_code, _json(fake)):
            self._add("Information Disclosure", "MEDIUM", "/api/login/safe",
                      "Different response for existing vs non-existing user",
                      f"{real.status_code} {_json(real)} vs {fake.status_code} {_json(fake)}")

    def run_all(self):
        """Run all DAST checks."""
        self.findings = []

        checks = [
            ("SQL Injection", self.test_sql_injection),
            ("XSS + SSTI", self.test_xss_and_ssti),
            ("SSRF", self.test_ssrf),
            ("CSRF", self.test_csrf),
            ("Info Disclosure", self.test_info_disclosure),
        ]

        for name, check_fn in checks:
            print(f"  Checking {name}...", end=" ")
            check_fn()
            print("done")

        return self.findings


# Run the DAST scanner
print("🔍 Gate 3: DAST — Scanning the running application")
print("=" * 70)

scanner = SimpleDAST(BASE_URL)
findings = scanner.run_all()

print(f"\n📊 Results: {len(findings)} vulnerabilities found\n")
for f in findings:
    icon = {"CRITICAL": "🔴", "HIGH": "🟠", "MEDIUM": "🟡"}.get(f["severity"], "⚪")
    print(f"  {icon} [{f['severity']}] {f['type']}")
    print(f"     Endpoint: {f['endpoint']}")
    print(f"     Evidence: {f['evidence']}")
    print()

critical = sum(1 for f in findings if f["severity"] == "CRITICAL")
high = sum(1 for f in findings if f["severity"] == "HIGH")

if critical > 0 or high > 0:
    print(f"❌ GATE FAILED: {critical} Critical, {high} High findings. Merge blocked.")
else:
    print("✅ GATE PASSED: No critical/high vulnerabilities found.")

# --- Is the scanner itself still working? -----------------------------------
# A DAST run that reports zero findings against a deliberately vulnerable app
# means the scanner broke, not that the app got safe.
found_types = {f["type"] for f in findings}
expected = {"SQL Injection", "Stored XSS", "Server-Side Template Injection",
            "SSRF", "Missing CSRF Protection", "Information Disclosure"}
assert expected <= found_types, (
    f"the scanner failed to reproduce: {sorted(expected - found_types)}"
)
regressions = [f for f in findings if f["endpoint"].endswith(("/safe", "/safe/"))]
assert not regressions, (
    "a /safe endpoint produced a finding — one of the fixes regressed:\n  "
    + "\n  ".join(f"{f['endpoint']}: {f['evidence']}" for f in regressions)
)
print("✅ scanner self-check: every expected vulnerability reproduced, and every")
print("   /safe control endpoint held.")

## Gate 4: Security Sign-Off Process

Automated tools catch most issues, but some require **human judgment**. At Microsoft, the SDL includes a formal security review before major releases.

### When Manual Review is Required

| Change Type | Automated Only? | Manual Review? |
|-------------|----------------|----------------|
| Bug fix in existing feature | ✅ Automated is enough | ❌ |
| New API endpoint | ✅ Automated | ⚠️ Recommended |
| New authentication flow | ✅ Automated | ✅ Required |
| Handling financial data | ✅ Automated | ✅ Required |
| Changes to encryption | ✅ Automated | ✅ Required |
| Third-party integration | ✅ Automated | ✅ Required |

### The security review checklist

A checklist made of nouns — *"threat model"*, *"logging"*, *"secrets"* — is a list of
topics, and everyone ticks every box. A useful checklist says **what to do, what
artifact proves it, and what makes it fail**, so two reviewers reach the same verdict:

| # | Do this | Evidence that satisfies it | Fails if |
|---|---------|---------------------------|----------|
| 1 | Re-open the threat model and diff it against this change | Updated DFD + threat table, dated after the design change | A new endpoint, data store, or outbound call is not on the diagram |
| 2 | Confirm every Critical/High threat has a named mitigation and a test | Threat table where each row links to a test or a control | Any Critical/High row has an empty mitigation, or a mitigation nobody tested |
| 3 | Run SAST on the diff | Bandit JSON report attached to the PR | Any HIGH finding without a written, reviewed `# nosec` justification |
| 4 | Run the dependency scan | `pip-audit` output attached | Any known CVE with no patched version pinned or documented exception |
| 5 | Run DAST against a deployed build | ZAP baseline report | Any Critical finding, or the scan did not actually reach the app (0 requests) |
| 6 | Check secrets handling in the diff | `git diff` reviewed for literals + secret-scan output | Any credential literal outside a test fixture; any secret read from a file committed to the repo |
| 7 | Confirm the change is observable | Log lines / metric names quoted in the PR description | A new auth or money path emits nothing an on-call engineer could search for |
| 8 | Name the rollback | One command or one toggle, written down | "We'd redeploy the previous version" with nobody having tried it |
| 9 | Pen test (major releases only) | Report from a team that did not write the code | Findings not tracked to closure, or no re-test after the fix |
| 10 | Sign-off | Named approver, dated, on the PR | Approved by the change's own author |

Note what row 5 rejects: *a scan that ran but reached nothing*. Green because it found
no problems and green because it never looked are indistinguishable in a summary line,
and the second is the one that ships breaches. This is the same trap the DAST scanner
above guards against with its `expected <= found_types` assertion, and the same one
the secret scanner in notebook 3 guards against by planting known secrets and checking
they come back. **Test your tests against a known-bad input, or you are trusting a
green check you never validated.**

In [ ]:
# === DEMO: Security Gate Pipeline Summary ===
# Combine all gates into a single pipeline report.
# Every number below comes from a gate we actually ran in this notebook — none of
# it is typed in by hand. A pipeline report with invented numbers is worse than no
# report: it reads exactly like a real one.

import datetime
import subprocess

try:
    branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"],
                            capture_output=True, text=True, timeout=5).stdout.strip()
except Exception:
    branch = ""

print("📋 Security Gate Pipeline Report")
print("=" * 70)
print(f"Application: Security Demo Flask App")
print(f"Scan Date:   {datetime.date.today().isoformat()}")
print(f"Branch:      {branch or '(not a git checkout)'}")
print()

# Gate 1 — the highest-severity SAST finding tells the reviewer where to start.
worst = sorted(
    bandit_results,
    key=lambda i: {"HIGH": 0, "MEDIUM": 1, "LOW": 2}[i["issue_severity"]],
)[0]
sast_action = (f"Start with {worst['test_id']} at line {worst['line_number']}: "
               f"{worst['issue_text'][:60]}")

# Gate 3 — group the DAST findings so the action names real endpoints.
dast_endpoints = sorted({f["endpoint"] for f in findings})

gates = [
    {
        "name": "Gate 1: SAST (Bandit)",
        "status": "FAIL" if bandit_high else "PASS",
        "details": f"{bandit_high} High, {bandit_medium} Medium, {bandit_low} Low",
        "action": sast_action if bandit_high else "None required",
    },
    {
        "name": "Gate 2: Dependency Scan (pip-audit)",
        "status": "PASS" if deps_clean else "FAIL",
        "details": ("no known vulnerabilities in the installed environment"
                    if deps_clean else "pip-audit reported known advisories"),
        "action": "None required" if deps_clean else "Upgrade the affected packages",
    },
    {
        "name": "Gate 3: DAST (automated scan)",
        "status": "FAIL" if (critical or high) else "PASS",
        "details": f"{len(findings)} findings — {critical} Critical, {high} High",
        "action": (f"Fix: {', '.join(dast_endpoints)}" if findings else "None required"),
    },
    {
        "name": "Gate 4: Security Review",
        "status": "PENDING",
        "details": "Awaiting security team review",
        "action": "Schedule review after automated gates pass",
    },
]

all_pass = True
for gate in gates:
    icon = {"PASS": "✅", "FAIL": "❌", "PENDING": "⏳"}[gate["status"]]
    print(f"{icon} {gate['name']}: {gate['status']}")
    print(f"   {gate['details']}")
    print(f"   Action: {gate['action']}")
    print()

    if gate["status"] == "FAIL":
        all_pass = False

print("=" * 70)
if all_pass:
    print("🎉 ALL GATES PASSED — Ready for merge!")
else:
    print("🚫 MERGE BLOCKED — Fix failing gates before merging.")
    print("\nThis is exactly how enterprise CI/CD pipelines work:")
    print("no code reaches production until ALL security gates pass.")

# This lab's app is deliberately broken, so the pipeline MUST block it. If this
# ever reports a clean run, the gates stopped measuring anything.
assert not all_pass, (
    "every gate passed against an intentionally vulnerable application — the "
    "pipeline is measuring nothing"
)
assert gates[0]["status"] == "FAIL" and gates[2]["status"] == "FAIL", (
    "expected both SAST and DAST to block this app"
)

## Example: GitHub Actions Security Pipeline

Here's what a real security gate pipeline looks like in GitHub Actions:

```yaml
# .github/workflows/security.yml
name: Security Gates

on:
  pull_request:
    branches: [main]

jobs:
  sast:
    name: "Gate 1: SAST (Bandit)"
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
      - run: pip install bandit

      # Step 1 — produce the report. Bandit exits non-zero when it finds
      # anything, so without continue-on-error the job dies here and the
      # artifact upload below never runs: you get a red X and no evidence.
      - name: Scan
        run: bandit -r app/ -f json -o bandit-report.json
        continue-on-error: true

      - uses: actions/upload-artifact@v4
        if: always()          # keep the evidence even when the gate fails
        with:
          name: bandit-report
          path: bandit-report.json

      # Step 2 — the gate itself, separate from the report, so the failure
      # message names the policy: HIGH severity blocks, everything else informs.
      - name: Gate on high-severity findings
        run: bandit -r app/ --severity-level high

  dependency-scan:
    name: "Gate 2: Dependency Scan"
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: astral-sh/setup-uv@v5   # `uv sync` needs uv on PATH first
        with:
          enable-cache: true
      - run: uv sync
      # Audit the LOCKED dependency set. A bare `pip-audit` would scan whatever
      # the runner's system Python happens to have, which is not your app.
      - run: uv run --with pip-audit pip-audit

  dast:
    name: "Gate 3: DAST (OWASP ZAP)"
    runs-on: ubuntu-latest
    needs: [sast, dependency-scan]  # only run if earlier gates pass
    steps:
      - uses: actions/checkout@v4
      # `docker-compose` (the v1 Python script) is no longer on GitHub runners.
      # Compose v2 is a docker subcommand. `--wait` blocks on healthchecks,
      # which beats `sleep 10` — that is a race dressed up as a delay.
      - run: docker compose up -d --wait
      - uses: zaproxy/action-baseline@v0.12.0
        with:
          target: "http://localhost:5001"

  security-review:
    name: "Gate 4: Security Sign-Off"
    runs-on: ubuntu-latest
    needs: [sast, dependency-scan, dast]
    environment: security-review  # requires manual approval
    steps:
      - run: echo "Security team has approved this release"
```

The `environment: security-review` line means a human from the security team must click "Approve" in GitHub before the pipeline continues.

### Three details that decide whether this pipeline is real

1. **Report and gate are separate steps.** Combine them and you must choose between
   failing the build and keeping the artifact. Reviewers need both.
2. **`--wait` only waits for services that declare a `healthcheck`.** Our
   `docker-compose.yml` has one for Postgres and Redis but not for `flask-app`, so
   Compose considers it up the moment the container starts — before Flask is
   listening. Add a healthcheck hitting `/health` before you trust this step, or
   ZAP will scan a connection-refused and report a clean run.
3. **Pin action versions to a tag you have checked.** `zaproxy/action-baseline`
   publishes new tags regularly; a version that doesn't exist fails the workflow
   loudly, but a floating `@master` fails quietly and differently every week.

## Penetration Testing

**Penetration testing** (pen testing) is when security experts manually try to break into your system. It goes beyond automated tools because humans can:
- Chain multiple low-risk issues into a critical exploit
- Think creatively about business logic flaws
- Find issues that automated tools miss

### Types of Pen Tests

| Type | Tester Knowledge | When to Use |
|------|-----------------|-------------|
| **Black box** | No knowledge of the system | Simulates an external attacker |
| **White box** | Full access to source code | Most thorough, finds the most issues |
| **Gray box** | Partial knowledge (e.g., API docs) | Balance of realism and thoroughness |

### Microsoft SDL Pen Testing Requirements

- Pen testing is **required** before major releases
- Must be done by an **independent team** (not the developers)
- Findings must be **tracked and remediated** before release
- Re-test after fixes to confirm they work

## 🔑 Key Takeaways

1. **Automate everything you can** — SAST, dependency scanning, and basic DAST should run on every PR
2. **SAST finds code-level bugs** early (Bandit for Python) — fast and catches SQL injection, hardcoded secrets
3. **Dependency scanning** catches vulnerabilities in your libraries — run `pip-audit` (recommended)
4. **DAST tests your running app** — sends real attacks to find runtime vulnerabilities
5. **Security gates block merges** — no code reaches production without passing all gates
6. **Human review** is still needed — for authentication changes, financial operations, and cryptography
7. **Penetration testing** validates everything before major releases
8. **Test your tests** — plant a known-bad input and check the tool reports it. A scan that reached nothing and a scan that found nothing produce the same green check, and only one of them is good news
9. **Automation is a floor, not a ceiling** — Bandit found this app's SQL injection and missed its XSS, SSTI, CSRF and SSRF. Passing gates means *no known pattern matched*, not *this code is safe*

### Enterprise Security Pipeline Summary

| Gate | Tool | Runs When | Blocks Merge? |
|------|------|-----------|---------------|
| SAST | Bandit, Semgrep, CodeQL | Every PR | ✅ Yes (High+ findings) |
| Dependencies | pip-audit, Dependabot | Every PR | ✅ Yes (known CVEs) |
| DAST | OWASP ZAP, Burp Suite | After deploy to staging | ✅ Yes (Critical findings) |
| Pen Test | Manual by security team | Before major releases | ✅ Yes (Critical findings) |
| Sign-Off | Security team review | Before production deploy | ✅ Yes (must be approved) |

## 🏁 Lab Complete!

You've learned the complete enterprise security review process:
1. **Threat Modeling** — identify what can go wrong (STRIDE)
2. **Vulnerability Knowledge** — understand and fix common attacks
3. **Secrets Management** — protect credentials properly
4. **Security Gates** — automate checks in CI/CD

These practices are used daily at Microsoft, Google, Amazon, and every major tech company.